# Akan (Asante Twi) TTS — ONE-CELL training (idempotent)
Author: Abdul Rashid Dickson · UG MSc stage 2 · repo: github.com/Dickson32-cell/akan-tts-asante

**Press the run button on the single cell below. That is the entire procedure.**
It installs, logs in, loads data, builds the training checkpoint, trains (resuming automatically), and pushes the model to the Hub when done.
If the session dies at any point: reconnect GPU, press the same button again.

In [ ]:
# ===================== AKAN TTS — ONE CELL, RUN-ME-REPEATEDLY =====================
import os, sys, json, glob, shlex, subprocess, time, pathlib
T0 = time.time()
def stage(msg): print('\n=== [%ds] %s ===' % (int(time.time()-T0), msg), flush=True)

# -- 0. GPU sanity -------------------------------------------------------------
stage('0/7 GPU check')
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit('NO GPU. Menu: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')
print('GPU:', r.stdout.strip())

# -- 1. Version pins (subprocess tools get these; kernel unaffected) -----------
stage('1/7 version pins (transformers 4.49 - harness requirement)')
os.system("pip install -q 'transformers==4.49.0' 'datasets==3.2.0' librosa soundfile accelerate > /dev/null 2>&1")
r = subprocess.run('python -c "import transformers,datasets; print(transformers.__version__, datasets.__version__)"', shell=True, capture_output=True, text=True)
print('subprocess env:', r.stdout.strip())
assert r.stdout.strip().split()[0].startswith('4.'), 'pin failed - re-run this cell'

# -- 2. Harness + Cython kernel (skip if already built) ------------------------
stage('2/7 training harness')
if not os.path.isdir('/content/finetune-hf-vits'):
    os.system('git clone -q https://github.com/ylacombe/finetune-hf-vits')
if not glob.glob('/content/finetune-hf-vits/monotonic_align/monotonic_align/*.so'):
    os.system('cd /content/finetune-hf-vits/monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace > /dev/null 2>&1')
print('harness ready:', os.path.isdir('/content/finetune-hf-vits'))

# -- 3. HF login (only asks once; token goes in the box, never in chat) --------
stage('3/7 Hugging Face login')
try:
    from huggingface_hub import whoami
    print('already logged in as:', whoami()['name'])
except Exception:
    from huggingface_hub import login
    login()   # paste your hf_ Write token in the box

# -- 4. Corpus: Drive manifest -> stream selected audio from the Hub -----------
stage('4/7 corpus (streaming ~9.4h audio; DONE in a previous run = skipped)')
if os.path.exists('/content/tts_data/train/dataset_info.json'):
    print('tts_data already built - skipping')
else:
    if not os.path.isdir('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    if not os.path.exists('/content/work/selected.jsonl'):
        os.system("mkdir -p /content/work && unzip -o -q '/content/drive/MyDrive/UG_TTS/tts_data.zip' -d /content/work")
    import urllib.request
    base = 'https://raw.githubusercontent.com/Dickson32-cell/akan-tts-asante/master/src/'
    for f in ['normalize_twi.py', 'extract_asante_colab.py']:
        pathlib.Path('/content/' + f).write_text(urllib.request.urlopen(base + f, timeout=60).read().decode('utf-8'), encoding='utf-8')
    rc = os.system('python /content/extract_asante_colab.py --manifest /content/work/selected.jsonl --out /content/tts_data')
    assert rc == 0, 'extraction failed - check output above, then re-run this cell'

# -- 5. Training checkpoint: aka generator + Meta aka discriminator ------------
stage('5/7 building training checkpoint (generator + discriminator)')
if glob.glob('/content/aka_train_base/*.safetensors') or glob.glob('/content/aka_train_base/*.bin'):
    print('aka_train_base already built - skipping')
else:
    rc = os.system('cd /content/finetune-hf-vits && python convert_original_discriminator_checkpoint.py --language_code aka --pytorch_dump_folder_path /content/aka_train_base')
    assert rc == 0, 'conversion failed - re-run this cell'
print('base ready:', sorted(os.path.basename(x) for x in glob.glob('/content/aka_train_base/*')))

# -- 5b. Patch harness dataset loading: load_from_disk (no format inference) ---
stage('5b/7 patching harness dataset loader')
_h = '/content/finetune-hf-vits/run_vits_finetuning.py'
_s = open(_h, encoding='utf-8').read()
if 'load_from_disk(data_args.dataset_name)' not in _s:
    import re as _re
    _pat_train = r'raw_datasets\[.train.\] = load_dataset\((?:.|\n)*?\)\n'
    _s = _re.sub(_pat_train, 'raw_datasets["train"] = load_from_disk(data_args.dataset_name)[data_args.train_split_name]\n', _s, count=1)
    _pat_eval = r'raw_datasets\[.eval.\] = load_dataset\((?:.|\n)*?\)\n'
    _s = _re.sub(_pat_eval, 'raw_datasets["eval"] = load_from_disk(data_args.dataset_name)[data_args.eval_split_name]\n', _s, count=1)
    if 'from datasets import DatasetDict, load_dataset' in _s:
        _s = _s.replace('from datasets import DatasetDict, load_dataset',
                        'from datasets import DatasetDict, load_dataset, load_from_disk')
    open(_h, 'w', encoding='utf-8').write(_s)
    print('harness PATCHED: load_from_disk')
_s = open(_h, encoding='utf-8').read()
_slice_old = 'batch[model_input_name] = string_inputs.get("input_ids")[: max_tokens_length + 1]'
if _slice_old in _s:
    _s = _s.replace(_slice_old,
        'batch[model_input_name] = string_inputs.get("input_ids")[: int(max_tokens_length) + 1]')
    open(_h, 'w', encoding='utf-8').write(_s)
    print('harness PATCHED: float-slice int cast')
else:
    print('harness slice already patched')

# -- 6. Config + launch (auto-resume if checkpoints exist) ---------------------
stage('6/7 training (loss lines scroll below; ~2-6h total; saves every 1000 steps)')
cfg = {
  'project_name': 'akan_twi_tts', 'do_train': True, 'do_eval': True,
  'push_to_hub': True,
  'hub_model_id': 'Dickson32-cell/akan-twi-mms', 'hub_strategy': 'checkpoint',
  'report_to': 'none', 'overwrite_output_dir': True,
  'output_dir': '/content/akan-vits-finetuned',
  'dataset_name': '/content/tts_data',
  'audio_column_name': 'audio', 'text_column_name': 'text',
  'train_split_name': 'train', 'eval_split_name': 'eval',
  'full_generation_sample_text': 'Mema wo akye, me nuanom. Wo ho te sɛn?',
  'max_duration_in_seconds': 12.0, 'min_duration_in_seconds': 2.0, 'max_tokens_length': 500,
  'model_name_or_path': '/content/aka_train_base',
  'preprocessing_num_workers': 1,   # 1 = in-process filtering; forking dies on Colab when audio ByteIO buffers are held (fork audit BufferError)
  'num_train_epochs': 30, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 8,
  'learning_rate': 0.0002, 'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.02,
  'adam_beta1': 0.8, 'adam_beta2': 0.99, 'weight_decay': 0.01,
  'eval_steps': 500, 'max_eval_samples': 25, 'save_steps': 1000, 'save_total_limit': 3,
  'weight_disc': 3, 'weight_fmaps': 1, 'weight_gen': 1, 'weight_kl': 1.5,
  'weight_duration': 1, 'weight_mel': 35,
  'fp16': True, 'seed': 456,
}
pathlib.Path('/content/finetune_akan.json').write_text(json.dumps(cfg, indent=1))

# bool style detection: Optional[bool] fields need explicit values
_harness_src = open('/content/finetune-hf-vits/run_vits_finetuning.py', encoding='utf-8').read()
_bool_value_style = ('do_train: Optional[bool]' in _harness_src) or ('do_eval: Optional[bool]' in _harness_src)
print('harness bool style:', 'value-required' if _bool_value_style else 'store-true')

flags = []
for k, v in cfg.items():
    f = '--' + k   # this harness defines its argparse flags WITH underscores
    if isinstance(v, bool):
        if _bool_value_style and v:
            flags += [f, 'True']
        elif v:
            flags.append(f)
    elif v is not None and v != '':
        flags += [f, str(v)]
cks = glob.glob('/content/akan-vits-finetuned/checkpoint-*')
if cks:
    flags += ['--resume_from_checkpoint', 'latest']
    print('RESUMING from %d existing checkpoints' % len(cks))
cmd_parts = 'cd /content/finetune-hf-vits && python run_vits_finetuning.py ' + shlex.join(flags) + ' > /content/train_full.log 2>&1'
proc = subprocess.Popen(cmd_parts, shell=True)
print('training launched (pid %s); streaming key lines below...' % proc.pid)
_last_len = 0
while proc.poll() is None:
    time.sleep(45)
    if not os.path.exists('/content/train_full.log'):
        continue
    _log = open('/content/train_full.log', encoding='utf-8', errors='replace').read()
    if len(_log) > _last_len:
        _new = _log[_last_len:]
        _last_len = len(_log)
        for line in _new.splitlines():
            ln = line.lower()
            if ('loss' in ln and 'epoch' in ln) or 'error' in ln or 'traceback' in ln or 'running training' in ln:
                print(line.strip(), flush=True)
print('training process exited rc=%s' % proc.returncode)
print('--- LAST 50 LINES OF train_full.log ---')
os.system('tail -50 /content/train_full.log')

# -- 7. Persist to Drive -------------------------------------------------------
stage('7/7 checkpoint -> Drive (UG_TTS)')
os.system("mkdir -p '/content/drive/MyDrive/UG_TTS/akan-finetuned'")
os.system("cp -r /content/akan-vits-finetuned/*.json '/content/drive/MyDrive/UG_TTS/akan-finetuned/' 2>/dev/null")
os.system("cp -r /content/akan-vits-finetuned/*.safetensors '/content/drive/MyDrive/UG_TTS/akan-finetuned/' 2>/dev/null")
os.system("cp -r /content/akan-vits-finetuned/*.bin '/content/drive/MyDrive/UG_TTS/akan-finetuned/' 2>/dev/null")
os.system("cp -r /content/akan-vits-finetuned/checkpoint-* '/content/drive/MyDrive/UG_TTS/akan-finetuned/' 2>/dev/null")
print('Drive copy done. If training was interrupted, press this cell again - it resumes.')
print('TOTAL elapsed: %ds' % int(time.time()-T0))
# ===============================================================================

### After the model appears on huggingface.co/Dickson32-cell/akan-twi-mms
Everything on the laptop side picks it up automatically (demo scripts, notebook, eval). Report the hub URL and the final training summary to the agent to proceed with evaluation, samples, report.